In [1]:
#加载分词器
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [7]:
#使用 tokenizer：
fenci=tokenizer("Using a Transformer network is simple")
print(fenci)

#保存 tokenizer：
tokenizer.save_pretrained("C:\\Users\\27729\\Desktop\\NLP-Learning\\outputs\\savetokenizer")

{'input_ids': [101, 7993, 170, 13809, 23763, 2443, 1110, 3014, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


('C:\\Users\\27729\\Desktop\\NLP-Learning\\outputs\\savetokenizer\\tokenizer_config.json',
 'C:\\Users\\27729\\Desktop\\NLP-Learning\\outputs\\savetokenizer\\special_tokens_map.json',
 'C:\\Users\\27729\\Desktop\\NLP-Learning\\outputs\\savetokenizer\\vocab.txt',
 'C:\\Users\\27729\\Desktop\\NLP-Learning\\outputs\\savetokenizer\\added_tokens.json',
 'C:\\Users\\27729\\Desktop\\NLP-Learning\\outputs\\savetokenizer\\tokenizer.json')

In [8]:
sequence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sequence)
print(tokens)

['Using', 'a', 'Trans', '##former', 'network', 'is', 'simple']


In [9]:
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[7993, 170, 13809, 23763, 2443, 1110, 3014]


In [10]:
tokens = tokenizer.tokenize("I've been waiting for a HuggingFace course my whole life.","I hate this so much!")
print(tokens)

['I', "'", 've', 'been', 'waiting', 'for', 'a', 'Hu', '##gging', '##F', '##ace', 'course', 'my', 'whole', 'life', '.', 'I', 'hate', 'this', 'so', 'much', '!']


In [11]:
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[146, 112, 1396, 1151, 2613, 1111, 170, 20164, 10932, 2271, 7954, 1736, 1139, 2006, 1297, 119, 146, 4819, 1142, 1177, 1277, 106]


In [12]:
decoded_string = tokenizer.decode([7993, 170, 13809, 23763, 2443, 1110, 3014])
print(decoded_string)

Using a Transformer network is simple


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
print("Tokens:", tokens)
ids = tokenizer.convert_tokens_to_ids(tokens)

input_ids = torch.tensor(ids)
#在这一步后面要加一个维度或者直接调用更高一级的封装tokenizer
#model(input_ids)# 这一行会运行失败

input_ids = torch.tensor([ids]) #变为了二维张量(1,14)
print("Input IDs:", input_ids)
output = model(input_ids)
print("Logits:", output.logits)

#tokenized_inputs = tokenizer(sequence, return_tensors="pt")


Tokens: ['i', "'", 've', 'been', 'waiting', 'for', 'a', 'hugging', '##face', 'course', 'my', 'whole', 'life', '.']
Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])
Logits: tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


In [21]:
#批处理
batched_ids = [ids, ids]
input_ids = torch.tensor(batched_ids)
print("Input IDs:", input_ids)
output = model(input_ids)
print("Logits:", output.logits)

Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012],
        [ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])
Logits: tensor([[-2.7276,  2.8789],
        [-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


In [22]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895],
        [ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


In [23]:
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],]

attention_mask = [
    [1, 1, 1],
    [1, 1, 0],]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)


tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


In [26]:
sequence1 = "I've been waiting for a HuggingFace course my whole life."
sequence2 = "I hate this so much!"
tokens1 = tokenizer.tokenize(sequence1)
tokens2 = tokenizer.tokenize(sequence2)
ids1 = tokenizer.convert_tokens_to_ids(tokens1)
ids2 = tokenizer.convert_tokens_to_ids(tokens2)
print("IDs for sequence 1:", ids1)
print("IDs for sequence 2:", ids2)
print(model(torch.tensor([ids1])).logits)
print(model(torch.tensor([ids2])).logits)

IDs for sequence 1: [1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012]
IDs for sequence 2: [1045, 5223, 2023, 2061, 2172, 999]
tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)
tensor([[ 3.1931, -2.6685]], grad_fn=<AddmmBackward0>)


In [27]:
#pading后批处理
batched_ids = [
    [1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012],
    [1045, 5223, 2023, 2061, 2172, 999, tokenizer.pad_token_id,tokenizer.pad_token_id,tokenizer.pad_token_id,tokenizer.pad_token_id,tokenizer.pad_token_id,tokenizer.pad_token_id,tokenizer.pad_token_id,tokenizer.pad_token_id],
]
print(model(torch.tensor(batched_ids)).logits)

tensor([[-2.7276,  2.8789],
        [ 2.5423, -2.1265]], grad_fn=<AddmmBackward0>)


In [28]:
#掩码后批处理
attention_mask = [
    [1, 1, 1,1,1,1,1,1,1,1,1,1,1,1,],
    [1, 1, 1,1,1,1,0,0,0,0,0,0,0,0,],]
outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

tensor([[-2.7276,  2.8789],
        [ 3.1931, -2.6685]], grad_fn=<AddmmBackward0>)
